In [ ]:
#!/usr/bin/env python3
"""
Calculate pdBSI (power difference Brain Symmetry Index) for Muse 2 EEG data.
Compares Rest, Language, and Emotional conditions across 4 frequency bands (TP9-TP10 only).
Power is averaged across the full recording first, then pdBSI is computed once per band.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FREQUENCY_BANDS = ['Delta', 'Theta', 'Alpha', 'Beta']
CONDITIONS      = ['rest', 'language', 'emotional']
ELECTRODE_PAIR  = ('TP9', 'TP10')

CONDITION_COLORS = {
    'rest':      '#4C72B0',
    'language':  '#DD8452',
    'emotional': '#55A868',
}

In [5]:
def guess_compression(file):
    """
    Guesses compression format for pandas based on filename suffix
    """
    l=str(file).lower()
    if l.endswith(".gz"):
        return "gzip"
    if l.endswith(".zip"):
        return "zip"
    return None

In [7]:
def process_condition(csv_file):
    """
    Load a CSV, average band power across the full recording,
    then compute a single pdBSI per band.
    """
    df = pd.read_csv(csv_file, compression=guess_compression(csv_file))
    df['TimeStamp'] = pd.to_datetime(df['TimeStamp'], errors='coerce')
    df = df[df['TimeStamp'].notna()].copy()
    print(f"  Loaded {len(df)} rows from {Path(csv_file).name}")

    left_elec, right_elec = ELECTRODE_PAIR
    band_stats = {}

    for band in FREQUENCY_BANDS:
        left_col  = f"{band}_{left_elec}"
        right_col = f"{band}_{right_elec}"

        if left_col not in df.columns or right_col not in df.columns:
            print(f"  Warning: {left_col} or {right_col} not found. Skipping.")
            continue

        left_power  = pd.to_numeric(df[left_col],  errors='coerce')
        right_power = pd.to_numeric(df[right_col], errors='coerce')

        # Average power across the full recording first, then compute pdBSI once
        mean_left  = np.nanmean(left_power)
        mean_right = np.nanmean(right_power)

        denom = mean_left + mean_right
        pdBSI_value = ((mean_right - mean_left) / denom) * 100 if denom != 0 else np.nan

        band_stats[band] = {'mean': pdBSI_value}
        print(f"    {band}: pdBSI = {pdBSI_value:.3f}%")

    return band_stats


def plot_comparison(all_results, participant):
    """
    Grouped bar chart: 4 frequency bands × 3 conditions, no error bars.
    """
    n_conditions = len(CONDITIONS)
    bar_width    = 0.22
    x            = np.arange(len(FREQUENCY_BANDS))
    offsets      = np.linspace(
                       -(n_conditions - 1) / 2,
                        (n_conditions - 1) / 2,
                        n_conditions) * bar_width

    fig, ax = plt.subplots(figsize=(11, 6))

    for i, condition in enumerate(CONDITIONS):
        means = [
            all_results[condition].get(band, {}).get('mean', np.nan)
            for band in FREQUENCY_BANDS
        ]
        positions = x + offsets[i]
        ax.bar(
            positions, means,
            width=bar_width,
            alpha=0.85,
            color=CONDITION_COLORS[condition],
            label=condition.capitalize(),
        )

    # Reference lines and shading
    ax.axhline(y=0,  color='black', linewidth=1)
    #ax.axhline(y=1,  color='gray',  linewidth=0.8, linestyle='--', alpha=0.6)
    #ax.axhline(y=-1, color='gray',  linewidth=0.8, linestyle='--', alpha=0.6)
    #ax.axhspan(-1, 1, color='lightyellow', alpha=0.35, zorder=0)

    ax.set_xticks(x)
    ax.set_xticklabels(FREQUENCY_BANDS, fontsize=12)
    ax.set_xlabel('Frequency Band', fontsize=12)
    ax.set_ylabel('pdBSI (%)', fontsize=12)
    ax.set_title(
        f'pdBSI by Frequency Band & Condition — TP9-TP10\n'
        f'Participant: {participant}',
        fontsize=13, fontweight='bold'
    )

    ymin, ymax = ax.get_ylim()
    ax.set_ylim(min(ymin, -1.4), max(ymax, 1.4))

    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(title='Condition', fontsize=10, title_fontsize=10)

    plt.tight_layout()
    plt.savefig(f"{participant}_pdBSI_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nPlot saved as: {participant}_pdBSI_comparison.png")


def get_summary_table(all_results):
    rows = []
    for condition, bands_data in all_results.items():
        for band, stats in bands_data.items():
            rows.append({
                'Condition':      condition.capitalize(),
                'Frequency_Band': band,
                'pdBSI':          round(stats['mean'], 3),
            })
    return pd.DataFrame(rows)

In [16]:
# ── Main ──────────────────────────────────────────────────────────────────────
data_dir = Path.cwd() / "data"
participant = "sample"

In [17]:
all_results = {}

for condition in CONDITIONS:
    for suffix in ["csv.gz", "zip"]:
        csv_path = data_dir / f"{participant}_{condition}.{suffix}"
        if csv_path.exists():
            break
    print(f"\nProcessing condition: {condition.upper()}")
    print(f"  File: {csv_path}")

    if not csv_path.exists():
        print(f"  ERROR: File not found — {csv_path}")
        continue

    all_results[condition] = process_condition(csv_path)

if all_results:
    summary = get_summary_table(all_results)
    print("\n── Summary Table ──────────────────────────────")
    print(summary.to_string(index=False))

    plot_comparison(all_results, participant)
else:
    print("No data loaded. Check file paths and filenames.")